# Notebook 08 — Unsupervised EEG State Discovery

This analysis discovers latent states from the 135 engineered EEG features without
using the pain/no-pain target. The target column is never loaded into the model
matrix and is not used to name or interpret states.

**Method**

1. Load the real 781-epoch feature table generated from 26 original BDF files.
2. Standardize the 135 EEG features.
3. Evaluate K-Means solutions with elbow and silhouette diagnostics.
4. Visualize the feature space with t-SNE.
5. Characterize each discovered state by feature family and strongest individual
   features.
6. Check whether state membership is dominated by particular subjects.
7. Evaluate KNN state assignment with subject-grouped cross-validation.

t-SNE is used only for visualization. Clustering is performed in the standardized
135-dimensional feature space because t-SNE does not preserve global distances.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, f1_score,
    normalized_mutual_info_score, silhouette_score
)
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data/preprocessed")

df = pd.read_csv(DATA_DIR / "all_subjects_features.csv")
stored = np.load(DATA_DIR / "features_xy.npz", allow_pickle=True)
dictionary = pd.read_csv(DATA_DIR / "feature_dictionary_135.csv")
feature_names = [str(x) for x in stored["feature_names"]]

assert len(df) == 781
assert len(feature_names) == 135
assert dictionary["feature_name"].tolist() == feature_names
assert df[feature_names].isna().sum().sum() == 0

X_raw = df[feature_names].to_numpy(dtype=float)
subjects = df["subject"].astype(str).to_numpy()
X = StandardScaler().fit_transform(X_raw)

print(f"Real epochs: {len(df):,}")
print(f"Subjects: {df['subject'].nunique()}")
print(f"EEG features used: {len(feature_names)}")
print("Target columns used in clustering: none")
print(f"Missing feature values: {int(np.isnan(X_raw).sum())}")

Real epochs: 781
Subjects: 26
EEG features used: 135
Target columns used in clustering: none
Missing feature values: 0


## Select the number of latent EEG states

The maximum silhouette score provides the deterministic selection rule. The elbow
curve is shown as an additional diagnostic.

In [2]:
candidate_k = range(2, 11)
rows = []
models = {}

for k in candidate_k:
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X)
    models[k] = model
    rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X, labels),
    })

selection = pd.DataFrame(rows)
best_k = int(selection.loc[selection["silhouette"].idxmax(), "k"])
selection.to_csv(DATA_DIR / "eeg_state_cluster_selection.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].plot(selection["k"], selection["inertia"], marker="o")
axes[0].axvline(best_k, color="black", linestyle="--", alpha=.5)
axes[0].set(title="Elbow diagnostic", xlabel="Number of states (k)", ylabel="Inertia")
axes[1].plot(selection["k"], selection["silhouette"], marker="o", color="#6a3d9a")
axes[1].axvline(best_k, color="black", linestyle="--", alpha=.5)
axes[1].set(title="Silhouette diagnostic", xlabel="Number of states (k)", ylabel="Silhouette")
fig.savefig("eeg_state_cluster_selection.png", dpi=180)
plt.show()

print(selection.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"Selected number of EEG states: {best_k}")

 k    inertia  silhouette
 2 74501.0741      0.2334
 3 65023.1175      0.2052
 4 58612.4184      0.1847
 5 53874.8829      0.1464
 6 50971.8244      0.1169
 7 49100.2132      0.1213
 8 47405.4937      0.1245
 9 46222.9377      0.0981
10 45298.9829      0.0990
Selected number of EEG states: 2


## t-SNE map and state assignments

The cluster model is fitted without labels. State IDs are arbitrary identifiers,
not ordered severity levels.

In [3]:
state_id = models[best_k].labels_

tsne = TSNE(
    n_components=2, perplexity=30, init="pca", learning_rate="auto",
    max_iter=1000, random_state=RANDOM_STATE
)
Z = tsne.fit_transform(X)

assignments = pd.DataFrame({
    "row_index": np.arange(len(df)),
    "subject": subjects,
    "tsne_1": Z[:, 0],
    "tsne_2": Z[:, 1],
    "eeg_state": state_id,
})
assignments.to_csv(DATA_DIR / "unsupervised_eeg_state_assignments.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
scatter = ax.scatter(
    Z[:, 0], Z[:, 1], c=state_id, cmap="tab10",
    s=16, alpha=.72, edgecolors="none"
)
ax.set(title=f"Unsupervised EEG states on t-SNE (k={best_k})",
       xlabel="t-SNE 1", ylabel="t-SNE 2")
ax.legend(*scatter.legend_elements(), title="EEG state", frameon=False)
fig.savefig("unsupervised_eeg_states_tsne.png", dpi=180)
plt.show()

print(assignments["eeg_state"].value_counts().sort_index().rename("epochs"))

eeg_state
0    163
1    618
Name: epochs, dtype: int64


## State composition and subject-dependence check

Subject IDs are used only here as an audit variable. High normalized mutual
information would indicate that clusters may reflect person-specific differences
rather than recurring within-person EEG states.

In [4]:
composition_rows = []
for state in sorted(np.unique(state_id)):
    mask = state_id == state
    counts = pd.Series(subjects[mask]).value_counts()
    composition_rows.append({
        "eeg_state": state,
        "epochs": int(mask.sum()),
        "subjects_represented": int(counts.size),
        "largest_subject_epochs": int(counts.iloc[0]),
        "largest_subject_fraction": float(counts.iloc[0] / mask.sum()),
    })

composition = pd.DataFrame(composition_rows)
subject_nmi = normalized_mutual_info_score(subjects, state_id)
composition["state_subject_nmi_overall"] = subject_nmi
composition.to_csv(DATA_DIR / "eeg_state_composition.csv", index=False)

subject_state = pd.crosstab(
    pd.Series(subjects, name="subject"),
    pd.Series(state_id, name="eeg_state"),
)
subject_state.to_csv(DATA_DIR / "eeg_state_by_subject_counts.csv")

print(composition.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"Normalized mutual information between subject and state: {subject_nmi:.4f}")
print("Lower values indicate less subject-specific clustering.")

 eeg_state  epochs  subjects_represented  largest_subject_epochs  largest_subject_fraction  state_subject_nmi_overall
         0     163                    15                      32                    0.1963                     0.1884
         1     618                    24                      32                    0.0518                     0.1884
Normalized mutual information between subject and state: 0.1884
Lower values indicate less subject-specific clustering.


## Feature profile of each state

Values below are cluster means in standard-deviation units relative to the full
cohort. Positive values mean that a feature is higher than the cohort mean; negative
values mean lower. Profiles describe the data and are not clinical state names.

In [5]:
profile = pd.DataFrame(
    [X[state_id == state].mean(axis=0) for state in sorted(np.unique(state_id))],
    index=[f"state_{state}" for state in sorted(np.unique(state_id))],
    columns=feature_names,
)
profile.index.name = "state"
profile.to_csv(DATA_DIR / "eeg_state_feature_profiles_z.csv")

long_profile = (
    profile.reset_index()
    .melt(id_vars="state", var_name="feature_name", value_name="mean_z")
    .merge(dictionary[["feature_name", "feature_family"]], on="feature_name", how="left")
)
family_profile = (
    long_profile.groupby(["state", "feature_family"], as_index=False)["mean_z"].mean()
)
family_profile.to_csv(DATA_DIR / "eeg_state_family_profiles_z.csv", index=False)

top_rows = []
for state in profile.index:
    values = profile.loc[state]
    for rank, feature in enumerate(values.abs().nlargest(12).index, start=1):
        meta = dictionary.loc[dictionary["feature_name"] == feature].iloc[0]
        top_rows.append({
            "state": state,
            "rank": rank,
            "feature_name": feature,
            "feature_family": meta["feature_family"],
            "mean_z": values[feature],
            "direction": "higher" if values[feature] > 0 else "lower",
        })
top_features = pd.DataFrame(top_rows)
top_features.to_csv(DATA_DIR / "eeg_state_top_features.csv", index=False)

pivot = family_profile.pivot(index="state", columns="feature_family", values="mean_z")
fig, ax = plt.subplots(figsize=(11, max(3, 1.2 * best_k)), constrained_layout=True)
image = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-.7, vmax=.7)
ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha="right")
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_title("Mean standardized feature-family profile by EEG state")
fig.colorbar(image, ax=ax, label="Mean z-score")
fig.savefig("eeg_state_family_profiles.png", dpi=180)
plt.show()

for state in profile.index:
    print(f"\n{state} strongest distinguishing features:")
    print(top_features[top_features["state"] == state][
        ["rank", "feature_name", "feature_family", "mean_z", "direction"]
    ].head(8).to_string(index=False, float_format=lambda x: f"{x:.3f}"))


state_0 strongest distinguishing features:
 rank     feature_name feature_family  mean_z direction
    1 C4_low_gamma_log     Band power   1.473    higher
    2 CZ_low_gamma_log     Band power   1.415    higher
    3 C3_low_gamma_log     Band power   1.399    higher
    4 CZ_low_gamma_abs     Band power   1.357    higher
    5      C4_beta_abs     Band power   1.313    higher
    6      C4_beta_log     Band power   1.276    higher
    7 C4_low_gamma_abs     Band power   1.227    higher
    8      C3_beta_abs     Band power   1.204    higher

state_1 strongest distinguishing features:
 rank     feature_name feature_family  mean_z direction
    1 C4_low_gamma_log     Band power  -0.389     lower
    2 CZ_low_gamma_log     Band power  -0.373     lower
    3 C3_low_gamma_log     Band power  -0.369     lower
    4 CZ_low_gamma_abs     Band power  -0.358     lower
    5      C4_beta_abs     Band power  -0.346     lower
    6      C4_beta_log     Band power  -0.337     lower
    7 C4_low_gam

## KNN recovery across held-out subjects

KNN is used to test whether the discovered state definitions can be assigned to
epochs from subjects excluded from training. Scaling is fitted separately inside
each training fold.

In [6]:
neighbor_candidates = [1, 3, 5, 7, 11, 15]
cv = GroupKFold(n_splits=5)
knn_rows = []

for n_neighbors in neighbor_candidates:
    fold_metrics = []
    for train_idx, test_idx in cv.split(X_raw, state_id, subjects):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_raw[train_idx])
        X_test = scaler.transform(X_raw[test_idx])
        model = KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance")
        model.fit(X_train, state_id[train_idx])
        pred = model.predict(X_test)
        fold_metrics.append({
            "accuracy": accuracy_score(state_id[test_idx], pred),
            "macro_f1": f1_score(state_id[test_idx], pred, average="macro"),
            "ari": adjusted_rand_score(state_id[test_idx], pred),
        })
    folds = pd.DataFrame(fold_metrics)
    knn_rows.append({
        "n_neighbors": n_neighbors,
        "grouped_accuracy_mean": folds["accuracy"].mean(),
        "grouped_macro_f1_mean": folds["macro_f1"].mean(),
        "grouped_ari_mean": folds["ari"].mean(),
    })

knn_results = pd.DataFrame(knn_rows)
best_neighbors = int(knn_results.loc[
    knn_results["grouped_macro_f1_mean"].idxmax(), "n_neighbors"
])
knn_results.to_csv(DATA_DIR / "knn_unsupervised_eeg_state_cv.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.plot(knn_results["n_neighbors"], knn_results["grouped_accuracy_mean"],
        marker="o", label="Accuracy")
ax.plot(knn_results["n_neighbors"], knn_results["grouped_macro_f1_mean"],
        marker="s", label="Macro F1")
ax.set(title="Grouped KNN recovery of EEG states",
       xlabel="Number of neighbors", ylabel="Score", ylim=(0, 1))
ax.set_xticks(neighbor_candidates)
ax.legend(frameon=False)
fig.savefig("knn_unsupervised_eeg_state_cv.png", dpi=180)
plt.show()

print(knn_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"Best KNN neighbor count by grouped macro F1: {best_neighbors}")

 n_neighbors  grouped_accuracy_mean  grouped_macro_f1_mean  grouped_ari_mean
           1                 0.9505                 0.9154            0.7690
           3                 0.9493                 0.9174            0.7733
           5                 0.9518                 0.9101            0.7656
           7                 0.9518                 0.9045            0.7558
          11                 0.9470                 0.8939            0.7343
          15                 0.9416                 0.8804            0.7020
Best KNN neighbor count by grouped macro F1: 3


## Generated outputs

- `data/preprocessed/unsupervised_eeg_state_assignments.csv`
- `data/preprocessed/eeg_state_cluster_selection.csv`
- `data/preprocessed/eeg_state_composition.csv`
- `data/preprocessed/eeg_state_by_subject_counts.csv`
- `data/preprocessed/eeg_state_feature_profiles_z.csv`
- `data/preprocessed/eeg_state_family_profiles_z.csv`
- `data/preprocessed/eeg_state_top_features.csv`
- `data/preprocessed/knn_unsupervised_eeg_state_cv.csv`
- `eeg_state_cluster_selection.png`
- `unsupervised_eeg_states_tsne.png`
- `eeg_state_family_profiles.png`
- `knn_unsupervised_eeg_state_cv.png`

These are statistical EEG feature states. Assigning physiological names such as
arousal, drowsiness, or motor activation would require additional validated labels
or synchronized behavioral measurements.